# P8 — KGQA Bench: chạy CGAR + PRISM 12 bước trên Colab

Notebook này đi đúng 12 bước của *Sổ tay thực nghiệm CGAR–PRISM*. Mỗi bước = 1 cell markdown (mục tiêu, output kỳ vọng, gate) + 1 cell code.
Mọi cell đều **bỏ qua nếu output đã tồn tại** (xoá thư mục `results/<bước>` để chạy lại). Thứ tự bắt buộc: 00 → 01 → 02 → 03/04 → 05 → 06 → 07..11.

| Bước | Nội dung | GPU | Thời gian ước lượng (A100) |
|---|---|---|---|
| 00 | Thiết lập, test | – | 5 phút |
| 01 | Dữ liệu + trần coverage | – | 10–20 phút |
| 02 | Baseline không học | – | 15 phút |
| 03 | Train CGAR | yes | 1.5–3 giờ |
| 04 | Train PRISM (oracle teacher) | yes | 40 phút |
| 05 | Teacher LLM thật → PRISM | yes | 1.5 giờ |
| 06 | Bảng chính 3 seed + sweep | yes | 6–9 giờ |
| 07 | Calibration & abstention | yes | 20 phút |
| 08 | LLM reader & hallucination | yes | 1–3 giờ / reader |
| 09 | Ablation | yes | 2–3 giờ |
| 10 | Transfer & drift | yes | 1 giờ |
| 11 | Report + checklist claim | – | 1 phút |

## 00 · Thiết lập và smoke test
**Mục tiêu:** xác nhận toàn bộ chuỗi code chạy được trước khi tốn GPU.
**Output kỳ vọng:** `33 passed`, `ALL PASSED` (9 test), thư mục smoke có `results.json`, `report.md`.
**Gate:** mọi test xanh. Nếu fail → lỗi là code, sửa trước.

In [ ]:
# ==== 00a. Mount Drive + đường dẫn (SỬA 2 DÒNG ĐẦU NẾU KHÁC) ====
from google.colab import drive; drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab'             # = thư mục colab/ local (chứa graph_rag/, paper/)
GRAPH_RAG  = f'{DRIVE_BASE}/graph_rag'                                       # chứa grl_rag.py, adagraph_cgar_v2.py, prism/, kgqa_bench/
RESULTS    = f'{GRAPH_RAG}/results'
CKPT       = f'{GRAPH_RAG}/ckpt'

import os, sys, json, subprocess, time, shutil, glob
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True   # 2 runtime cùng import từ Drive → không ghi .pyc
for d in glob.glob(f'{GRAPH_RAG}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
os.chdir(GRAPH_RAG); sys.path.insert(0, GRAPH_RAG)
print('cwd =', os.getcwd())
assert os.path.exists('kgqa_bench/cli.py'), 'kgqa_bench/ chưa được sync lên Drive'

In [ ]:
# ==== 00b. Cài deps + GPU ====
!pip install -q datasets sentence-transformers transformers accelerate scikit-learn networkx tqdm matplotlib
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# HF token (chỉ cần cho Llama gated). Thêm secret HF_TOKEN trong Colab → Secrets.
try:
    from google.colab import userdata
    if userdata.get('HF_TOKEN'): os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
except Exception: print('(no HF_TOKEN secret — dùng mirror NousResearch cho Llama)')

In [ ]:
# ==== 00c. Test + smoke run (CPU, ~3 phút) ====
!python test_cgar_v2.py | tail -3
!python tests/test_kgqa_bench.py 2>&1 | grep -E "  ok |  FAIL |PASSED|FAILED"
!python tests/test_hallucination.py 2>&1 | grep -E "  ok |  FAIL |PASSED|FAILED"
!python tests/test_escalation.py 2>&1 | grep -E "  ok |  FAIL |PASSED|FAILED"
!python tests/test_v3_and_analysis.py 2>&1 | grep -E "  ok |  FAIL |PASSED|FAILED"
SMOKE = f'{RESULTS}/b00_smoke'
if not os.path.exists(f'{SMOKE}/results.json'):
    cmd = ('python -m kgqa_bench.cli run --train synthetic:60:0:train --eval synthetic:30:1 '
           '--methods gnnrag_all gnnrag_top5 graphrag_fi cgar prism --embedder hash --device cpu '
           '--cgar-s1-epochs 2 --cgar-s2-steps 10 --prism-distill-epochs 2 --prism-rl-steps 10 '
           f'--prism-d-model 32 --prism-layers 1 --reader dummy --quiet --out {SMOKE}')
    print('exit =', subprocess.call(cmd, shell=True))
print(open(f'{SMOKE}/report.md').read()[:1500])

In [ ]:
# ==== 00d. Helper dùng chung cho các bước sau ====
import pandas as pd

def run_bench(out, train=None, evals=(), methods=(), extra='', force=False):
    """Gọi CLI; bỏ qua nếu results.json đã có (force=True để chạy lại)."""
    if os.path.exists(f'{out}/results.json') and not force:
        R = load_results(out)
        have = {m for by_m in R['summaries'].values() for m in by_m}
        missing = [m for m in methods if not any(h == m or h.startswith(m + '_k') for h in have)]
        if not missing:
            print(f'[skip] {out} đã có results.json'); return R
        print(f'[rerun] {out} thiếu {missing} → chạy lại (bản cũ đổi tên *_old)')
        old = f'{out}_old'; i = 1
        while os.path.exists(old): i += 1; old = f'{out}_old{i}'
        os.rename(out, old)
    cmd = f'python -m kgqa_bench.cli run --device {DEVICE} --out {out} '
    if train: cmd += f'--train {train} '
    cmd += '--eval ' + ' '.join(evals) + ' --methods ' + ' '.join(methods) + ' ' + extra
    print('$', cmd); t0 = time.time()
    rc = subprocess.call(cmd, shell=True)
    print(f'exit={rc}  {(time.time()-t0)/60:.1f} min')
    return load_results(out)

def load_results(out):
    with open(f'{out}/results.json') as f: return json.load(f)

COLS = ['n','f1','cov','prec','hit_ctx','cost','n_tokens','latency_ms','ece','aurc','sel_acc@80','las_hit1','llm_hit','llm_f1','llm_unsupported_rate']
def show(R, cols=COLS):
    for ds, by_m in R['summaries'].items():
        df = pd.DataFrame(by_m).T
        df = df[[c for c in cols if c in df.columns]]
        print(f'\n=== {ds} ==='); display(df.round(3))

def show_cmp(R):
    rows = []
    for ds, pairs in R['comparisons'].items():
        for k, c in pairs.items():
            if not c or not c.get('n'): continue
            b = c['bootstrap']
            rows.append(dict(dataset=ds, pair=k, n=c['n'], delta=round(b['delta']*100,2),
                             ci=f"[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}]", p=round(b['p_value'],4),
                             mcnemar_p=round(c.get('mcnemar',{}).get('p_value', float('nan')),4), sig=c['significant_0.05']))
    display(pd.DataFrame(rows))

def gate(cond, msg):
    print(('PASS  ' if cond else 'FAIL  ') + msg)

# Pool ứng viên WebQSP: USE_GNN = 'auto' dùng features GNN-RAG (ReaRev) nếu P8b cell 01 đã tạo
# ckpt/gnn_features_*.pkl; True = bắt buộc GNN; False = heuristic (RoG HF) cho vòng ablation.
USE_GNN = 'auto'
_tr, _te = f'{CKPT}/gnn_features_train.pkl', f'{CKPT}/gnn_features_webqsp_test.pkl'
if USE_GNN == 'auto': USE_GNN = os.path.exists(_tr) and os.path.exists(_te)
if USE_GNN:
    assert os.path.exists(_tr) and os.path.exists(_te), 'thiếu ckpt/gnn_features_*.pkl → chạy P8b cell 01 trước'
    WEBQSP_TR = f'gnnrag:{_tr}:webqsp:train'
    WEBQSP_TE = f'gnnrag:{_te}:webqsp:test'
else:
    WEBQSP_TR = 'rog:rmanluo/RoG-webqsp:train'
    WEBQSP_TE = 'rog:rmanluo/RoG-webqsp:test'
# CWQ (C2): dùng features ReaRev nếu P8b đã dựng ckpt/gnn_features_cwq_test.pkl, ngược lại heuristic (các dòng CWQ chỉ để minh hoạ pool)
_cwq = f'{CKPT}/gnn_features_cwq_test.pkl'
CWQ_TE = f'gnnrag:{_cwq}:cwq:test' if os.path.exists(_cwq) else 'rog:rmanluo/RoG-cwq:test'
# PRISM: 'auto' = quyết định ở cell 04 bằng gate (LAS trên mức chance, cost không hằng); False = nhánh CGAR-only
# (bỏ 05, phần PRISM của 06–10); True = bắt buộc chạy PRISM đầy đủ.
RUN_PRISM = False   # vòng 11/2026: CGAR-only; PRISM oracle đã qua gate (las 0.661 vs chance 0.451) → chạy vòng riêng sau nếu cần ('auto'/True)
# Budget đường của CGAR: 5 trên pool GNN (cost ≈ 5.1 như Seminar 1; pool chỉ ~5.6 ứng viên nên 10 ≈ "all"),
# 10 trên pool heuristic (như lần 11/2026). Dùng thống nhất ở cell 03, 06, 09, 10.
CGAR_BUDGET = 5 if WEBQSP_TR.startswith('gnnrag:') else 10
# ---- Cải tiến 11/2026 (xem paper/CGAR_PRISM_models_flow_results_2026-11.md §12) ----
CGAR_S2_STEPS = 0            # B4: S1 supervised là phương pháp chính (RL +0.19 pp, p=0.033 → vào ablation 09)
# B1 learned stop + B2 nhãn proof + B3 confidence theo hit. Cell 03 train cả bản gốc lẫn v3 và CHỌN TRÊN DEV;
# bản được chọn dùng cho 06/08/10 qua CGAR_FLAGS. Đặt CGAR_VARIANT = None để bỏ qua v3.
CGAR_VARIANT = '--cgar-learned-stop --cgar-rel-label proof --cgar-conf-target hit'
CGAR_FLAGS = ''              # cell 03 đặt lại ('' = gốc, CGAR_VARIANT = v3)
READER_PROMPTS = ['chat', 'grounded']   # D1: ma trận prompt gốc × grounded (chỉ trả lời từ paths, cho phép 'unknown')
EMB_MODEL = 'all-MiniLM-L6-v2'   # = embedder mặc định của CLI (--embedder sbert); cell 07/10 phải dùng cùng model
SPEC_TAG = 'gnn' if WEBQSP_TR.startswith('gnnrag:') else 'heur'
RESULTS  = f'{GRAPH_RAG}/results' + ('_gnn' if SPEC_TAG == 'gnn' else '')   # tách kết quả theo loại pool ứng viên
os.makedirs(RESULTS, exist_ok=True)
print('helpers ready | pool =', SPEC_TAG, '| RESULTS =', RESULTS, '| CGAR_BUDGET =', CGAR_BUDGET, '| CWQ =', 'gnn' if CWQ_TE.startswith('gnnrag:') else 'heur'); print('WEBQSP_TR =', WEBQSP_TR)

In [ ]:
# ==== 00e. (Tuỳ chọn) Dùng lại kết quả P8b đã chạy trên cùng pool GNN ====
# P8b cell 02/03 chạy đúng lệnh CLI như P8 cell 02/03 (pool 20, --cgar-max-paths 5) → copy sang để run_bench bỏ qua.
# Chỉ copy khi results.json đã có (CLI ghi file này ở cuối) → an toàn kể cả khi P8b đang train dở.
if SPEC_TAG == 'gnn':
    for g, b in [('g02_baselines', 'b02_baselines'), ('g03_cgar', 'b03_cgar')]:
        src, dst = f'{GRAPH_RAG}/results/{g}', f'{RESULTS}/{b}'
        if os.path.exists(dst): print(f'[có sẵn] {dst}')
        elif os.path.exists(f'{src}/results.json'): shutil.copytree(src, dst); print(f'[copy] {g} → {dst}')
        else: print(f'[chưa xong] P8b {g} chưa có results.json → cell tương ứng của P8 sẽ tự chạy (hoặc chạy lại cell này sau)')
    # cache CWQ heuristic (3531 câu, ~10 phút) dùng chung được
    os.makedirs(f'{RESULTS}/data_cache', exist_ok=True)
    for f in glob.glob(f'{GRAPH_RAG}/results/data_cache/cwq_*.pkl'):
        d = f'{RESULTS}/data_cache/{os.path.basename(f)}'
        if not os.path.exists(d): shutil.copy(f, d); print('[copy cache]', os.path.basename(f))
else:
    print('[skip] pool heuristic: P8b không có gì để dùng lại')

## 01 · Dữ liệu và trần coverage
**Mục tiêu:** nạp WebQSP/CWQ về dạng chung; biết `answer_in_subgraph_rate` = recall tối đa của *mọi* selector.
**Output kỳ vọng:** WebQSP test 1628 câu, CWQ 3531. Trần subgraph RoG: 0.956 / 0.807 (đo 11/2026), mean_hops 1.3 / 1.43, ~4.3k triple mỗi câu. Với features GNN-RAG (P8b): trần ≈0.73.
**Gate:** trần RoG ≥ 0.9 hoặc trần GNN trong [0.65, 0.85] tuỳ nguồn pool. Khoảng cách giữa trần subgraph (0.96) và hit của pool 20 ứng viên (0.33 ở bước 02) chính là phần scorer heuristic làm mất.

In [ ]:
from kgqa_bench.data import load_spec
DATA_CACHE = f'{RESULTS}/data_cache'
stats = {}
for spec in (WEBQSP_TR, WEBQSP_TE, CWQ_TE):
    ds = load_spec(spec, cache_dir=DATA_CACHE)
    stats[ds.key] = ds.stats(); print(ds.key, stats[ds.key])
display(pd.DataFrame(stats).T.round(3))
ceil = stats['webqsp:test']['answer_in_subgraph_rate']; src = str(stats['webqsp:test']['source'])
if src.startswith('gnnrag_pkl'):
    gate(0.80 <= ceil <= 0.95, f'trần subgraph WebQSP(GNN features) = {ceil:.3f} trong [0.80, 0.95] (11/2026: 0.887)')
    import numpy as np
    te_ = load_spec(WEBQSP_TE, cache_dir=DATA_CACHE)
    ph = float(np.mean([any(a in set(q.top_candidates) for a in q.answers) for q in te_ if q.answers]))
    gate(0.65 <= ph <= 0.85, f'đáp án trong pool ứng viên ReaRev (test) = {ph:.3f} (Seminar 1 ≈ 0.73; heuristic ≈ 0.33)')
else:
    gate(ceil >= 0.9, f'trần coverage WebQSP(subgraph RoG) = {ceil:.3f} ≥ 0.9 (lần 11/2026: 0.956; CWQ 0.807)')
    print('Lưu ý: đây là trần của SUBGRAPH. Pool 20 ứng viên heuristic chỉ chứa đáp án ở ~33% câu (hit_ctx gnnrag_all ở bước 02) → scorer là nút thắt, không phải dữ liệu.')
json.dump(stats, open(f'{RESULTS}/b01_dataset_stats.json','w'), indent=2)

## 02 · Baseline không học
**Mục tiêu:** đường cơ sở cost–quality và các bất đẳng thức bắt buộc: Rec(all) > Rec(top10) > Rec(top5) > Rec(top3); Rec(random_k5) < Rec(top5).
**Output kỳ vọng:** cost top-K đúng bằng K; GraphRAG-FI nằm giữa top5 và top10. Mọi baseline dùng chung pool 20 ứng viên với CGAR (`--baseline-candidates 20`, mặc định). `sbert_topK` = xếp đường theo cosine(câu hỏi, đường) bằng chính embedder của CGAR, không học: đây là mốc CGAR phải vượt rõ.
**Gate:** random_k5 thua top5 với CI không chồng. Nếu random ≈ top-K → scorer không có tín hiệu: thêm `--score-sim 0.5` hoặc dùng GNN-RAG features thật.

In [ ]:
B02 = f'{RESULTS}/b02_baselines'
R02 = run_bench(B02, evals=[WEBQSP_TE, CWQ_TE],
                methods=['gnnrag_all','gnnrag_top10','gnnrag_top5','gnnrag_top3','sbert_top10','sbert_top5','graphrag_fi','random_k5'],
                extra='--compare-ref gnnrag_top5 --quiet')
show(R02)
s = R02['summaries']['webqsp:test']
gate(s['gnnrag_all']['cov'] > s['gnnrag_top10']['cov'] > s['gnnrag_top5']['cov'] > s['gnnrag_top3']['cov'], 'Rec giảm đơn điệu theo K')
gate(s['random_k5']['hit_ctx_ci_high'] < s['gnnrag_top5']['hit_ctx_ci_low'], 'random_k5 < top5 (CI không chồng)')
TOPK_AT_COST5 = s['gnnrag_top5']['f1']; print('mốc F1 top-5 (cost 5) =', round(TOPK_AT_COST5*100, 2))
SBERT_AT_COST10 = s['sbert_top10']['f1']; SBERT_AT_COST5 = s['sbert_top5']['f1']
print('mốc F1 sbert_top10 (cost 10) = %.2f | sbert_top5 (cost 5) = %.2f' % (SBERT_AT_COST10*100, SBERT_AT_COST5*100))
gate(s['sbert_top5']['hit_ctx'] > s['gnnrag_top5']['hit_ctx'] or s['random_k5']['hit_ctx_ci_high'] < s['gnnrag_top5']['hit_ctx_ci_low'], 'ít nhất một ranking (GNN hoặc SBERT) có tín hiệu so với random')

## 03 · Huấn luyện CGAR
**Mục tiêu:** controller CGAR chọn checkpoint trên **dev tách từ train** (không phải test). Đo S2 (RL) có học không.
**Output kỳ vọng:** `rel_loss` giảm từ ≈0.69 (= log 2) xuống <0.5; `eval_f1` tăng rồi phẳng; `advantage_nonzero_frac` gần 0 (Seminar 1). WebQSP test F1 ≈40 / Rec ≈69 / cost ≈5 với GNN features thật, thấp hơn vài điểm với scorer heuristic.
**Budget:** `--cgar-max-paths` = `CGAR_BUDGET` (5 trên pool GNN như Seminar 1, 10 trên heuristic), dùng thống nhất ở 03/06/08/09/10. **Gate:** eval_f1 dev ≥ F1 top-K cùng cost; rel_loss < 0.6; F1 CGAR ≥ sbert_topK cùng cost + 1 pp. Mục *Adaptivity* trong report.md phải cho thấy cost/chosen_k thay đổi theo câu; nếu "constant" → bật `--cgar-rel-threshold`.

In [ ]:
B03 = f'{RESULTS}/b03_cgar'
# bản gốc v2.1 (S1; S2 chỉ khi CGAR_S2_STEPS>0). Thư mục cũ có S2=300 vẫn được dùng lại (ckpt S1 giống nhau).
R03 = run_bench(B03, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['cgar'],
                extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s1-epochs 10 --cgar-s2-steps {CGAR_S2_STEPS} --compare-ref cgar --quiet')
show(R03)
log = R03['train_logs']['cgar@seed42']
h1 = pd.DataFrame(log['stage1']['history']); display(h1[['epoch','loss','rel_loss','eval_f1','eval_cost']].round(4))
if log.get('stage2'):
    print('S2 advantage_nonzero_frac =', round(log['stage2'].get('advantage_nonzero_frac', float('nan')), 3))
    print('S2 eval_f1 history:', [(r['step'], round(r['eval_f1']*100,2)) for r in log['stage2'].get('history', [])])
gate(h1['rel_loss'].iloc[-1] < 0.6, 'rel_loss < 0.6 (gradient tới relevance head)')
gate(R03['summaries']['webqsp:test']['cgar']['f1'] >= TOPK_AT_COST5 - 0.005, 'F1 CGAR ≥ top-5 cùng cost')
_c = R03['summaries']['webqsp:test']['cgar']; _ref = SBERT_AT_COST10 if _c['cost'] > 7.5 else SBERT_AT_COST5
gate(_c['f1'] >= _ref + 0.01, f"F1 CGAR {_c['f1']:.3f} ≥ sbert_topK cùng cost {_ref:.3f} + 1 pp (gain đến từ controller, không chỉ embedding)")
print(open(f'{B03}/report.md').read().split('## Adaptivity')[-1][:600] if 'Adaptivity' in open(f'{B03}/report.md').read() else 'no adaptivity section')
CGAR_CKPT = f'{B03}/ckpt/cgar_s1_seed42.pt'   # B4: S1 là phương pháp chính (RL vào ablation 09)
CGAR_FLAGS = ''; DEV_BASE = log['stage1']['best_eval_f1']
# ---- v3 (B1+B2+B3): train cùng protocol, CHỌN TRÊN DEV (không nhìn test) ----
if CGAR_VARIANT:
    B03V = f'{RESULTS}/b03_cgar_v3'
    R03V = run_bench(B03V, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['cgar'],
                     extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s1-epochs 10 --cgar-s2-steps 0 {CGAR_VARIANT} --compare-ref cgar --quiet')
    logv = R03V['train_logs']['cgar@seed42']; DEV_V3 = logv['stage1']['best_eval_f1']
    hv = pd.DataFrame(logv['stage1']['history']); display(hv[['epoch','loss','rel_loss','eval_f1','eval_cost']].round(4))
    sv, sb = R03V['summaries']['webqsp:test']['cgar'], R03['summaries']['webqsp:test']['cgar']
    print(f"dev F1: gốc {DEV_BASE*100:.2f} | v3 {DEV_V3*100:.2f}   (test chỉ để tham khảo: gốc {sb['f1']*100:.2f}@{sb['cost']:.2f} ECE {sb['ece']:.3f} AURC {sb['aurc']:.3f} | v3 {sv['f1']*100:.2f}@{sv['cost']:.2f} ECE {sv['ece']:.3f} AURC {sv['aurc']:.3f})")
    gate(sv['cost'] < sb['cost'] - 0.1, f"v3 learned-stop tự giảm cost ({sv['cost']:.2f} < {sb['cost']:.2f}) mà không cần rel_threshold cố định")
    gate(sv['ece'] < sb['ece'], f"v3 conf-target=hit giảm ECE ({sv['ece']:.3f} < {sb['ece']:.3f})")
    if DEV_V3 >= DEV_BASE - 0.002:   # chọn v3 khi không thua trên dev quá 0.2 pp (v3 mang thêm calib + stop)
        CGAR_FLAGS, CGAR_CKPT = CGAR_VARIANT, f'{B03V}/ckpt/cgar_s1_seed42.pt'
        print('→ CHỌN v3 cho 06/08/10 (dev không thua)')
    else:
        print('→ GIỮ bản gốc cho 06/08/10 (v3 thua trên dev); v3 vẫn được báo cáo ở 09')
print('CGAR_CKPT =', CGAR_CKPT, '| CGAR_FLAGS =', repr(CGAR_FLAGS))

## 04 · Huấn luyện PRISM với teacher oracle
**Mục tiêu:** LAS + controller chạy trên dữ liệu thật, đạt mức sàn trước khi trả tiền cho teacher LLM. Nhãn kết quả: `oracle_heuristic` (KHÔNG được gọi là distilled from LLM).
**Output kỳ vọng:** cost phân bố 3–10 theo câu; `las_hit1` rõ ràng trên mức chance 1/32. Lần heuristic 11/2026: las_hit1 0.029 (= chance), cost luôn 3, eps-sweep phẳng → PRISM không học được gì trên pool heuristic; chỉ có ý nghĩa với GNN features.
**Gate:** las_hit1 ≥ chance + 10 pp (chance = mean 1/n_candidates; pool GNN ≈ 0.45 vì nhiều câu chỉ 1–2 candidate); cost không hằng; PRISM ≥ top-3 cùng cost. Lần GNN 11/2026: las_hit1 0.661 vs chance 0.451, cost {1,2,3}, F1 0.367@2.7 ≥ top-3 0.351@2.45 → qua cửa (bản cũ in nhãn '2× chance' nhưng so với 0.90 → đánh rớt oan, RUN_PRISM=False).

In [ ]:
if RUN_PRISM is False:
    print('[skip] RUN_PRISM = False → bỏ bước 04'); PRISM_ORACLE_CKPT = None
else:
    B04 = f'{RESULTS}/b04_prism_oracle'
    R04 = run_bench(B04, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['prism'],
                    extra='--dev-split 0.1 --prism-distill-epochs 10 --prism-rl-steps 500 --compare-ref prism --quiet')
    show(R04)
    log = R04['train_logs']['prism@seed42']; print('teacher =', log['teacher'], '| vocab =', log['vocab'])
    recs = [json.loads(l) for l in open(f'{B04}/records_webqsp_test_prism.jsonl')]
    from collections import Counter; print('stopped_reason:', Counter(r['stopped_reason'] for r in recs))
    costs = Counter(round(r['cost']) for r in recs); print('cost histogram:', costs, '| mean_step_kl = %.4f' % R04['summaries']['webqsp:test']['prism'].get('mean_step_kl', float('nan')))
    s4 = R04['summaries']['webqsp:test']['prism']
    import numpy as np
    chance = float(np.mean([1 / max(r.get('n_candidates') or 32, 1) for r in recs]))   # đoán mò trong pool: 1/32 heuristic, ≈1/5.6 GNN
    PRISM_CHANCE = chance
    # Pool GNN có nhiều câu chỉ 1–2 candidate nên chance (mean 1/n_cand) cao (~0.45) → dùng margin tuyệt đối thay vì 2× chance
    _las_ok = s4['las_hit1'] >= chance + 0.10
    gate(_las_ok, f"LAS hit@1 = {s4['las_hit1']:.3f} ≥ chance ({chance:.3f}) + 10 pp — LAS học được (heuristic 11/2026: 0.029 = chance; GNN 11/2026: 0.661 vs chance 0.451)")
    gate(len(costs) > 1, 'cost PRISM thay đổi theo câu (lần heuristic: luôn = 3 vì KL bước đầu < eps → dừng ngay)')
    s3 = R02['summaries']['webqsp:test']['gnnrag_top3']
    gate(s4['f1'] >= s3['f1'] - 0.005 and s4['hit_ctx'] >= s3['hit_ctx'], 'PRISM(oracle) ≥ top-3 cùng cost ≈ 3 (F1 và hit_ctx)')
    PRISM_ORACLE_CKPT = f'{B04}/ckpt/prism_seed42.pt'
    if RUN_PRISM == 'auto':
        RUN_PRISM = bool(_las_ok and len(costs) > 1)
        print('RUN_PRISM =', RUN_PRISM, '(auto: LAS trên mức chance và cost không hằng)' if RUN_PRISM else '(auto: PRISM không qua cửa → nhánh CGAR-only, bỏ 05 và phần PRISM của 06–10)')

## 05 · Teacher LLM thật cho PRISM
**Mục tiêu:** thí nghiệm quyết định claim trung tâm: retriever mô phỏng LLM. So PRISM(LLM) với PRISM(oracle) của bước 04.
**Output kỳ vọng:** cache ≈ 3000 câu × 6 subset entry; hit_rate cache > 0.8; eval_kl 0.2–0.4; F1 chênh ±2 điểm so với oracle.
**Gate / kill criterion:** ΔF1(LLM − oracle) ≥ −1 → dùng checkpoint này cho mọi bảng. Kém >3 điểm hoặc KL > 0.5 → PRISM về future work, CGAR làm chính.
Cache được dựng **trực tiếp từ sample của harness** để khớp key (qid, tập path).

In [ ]:
if not RUN_PRISM:
    print('[skip] RUN_PRISM = False → bỏ bước 05'); PRISM_CKPT = None; TEACHER_CACHE = None
else:
    TEACHER_CACHE = f'{CKPT}/teacher_logits_webqsp_{SPEC_TAG}.pt'   # key cache = (qid, tập path) → phải tách theo pool
    _old = f'{CKPT}/teacher_logits_webqsp_bench.pt'                    # cache heuristic đã dựng 11/2026 (18k entry, Llama-3.1-8B)
    if SPEC_TAG == 'heur' and not os.path.exists(TEACHER_CACHE) and os.path.exists(_old): TEACHER_CACHE = _old
    TEACHER_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'   # fallback tự động: NousResearch/Meta-Llama-3.1-8B-Instruct
    if not os.path.exists(TEACHER_CACHE):
        from kgqa_bench.data import load_spec
        from kgqa_bench.adapters import PrismVocab
        from kgqa_bench.methods import PRISMMethod
        from grl_rag import Embedder
        from prism.config import CFG
        from prism.teacher_llm import LiveTeacherLLM, TeacherConfig, build_teacher_cache
        tr = load_spec(WEBQSP_TR, cache_dir=DATA_CACHE)        # theo spec hiện hành (rog: hoặc gnnrag:)
        n_dev = int(len(tr) * 0.1); tr.questions = tr.questions[:-n_dev]          # cùng cách carve dev như CLI
        emb = Embedder(model_name=EMB_MODEL); vocab = PrismVocab().fit(tr)
        pm = PRISMMethod(emb, vocab=vocab, cfg=CFG, device=DEVICE)
        samples = pm.samples_for(tr, force_gold=True)[:3000]
        rel_lookup = {v: k for k, v in vocab.rel2id.items()}
        ent_lookup = {}; q_lookup = {}
        for s in samples:
            q_lookup[s.qid] = s.question_text
            for cid, name in zip(s.candidate_ids.tolist(), s.candidate_names): ent_lookup[cid] = name
        teacher = LiveTeacherLLM(TeacherConfig(model_name=TEACHER_MODEL, scoring_mode='letter'))
        class _L:  # list-like wrapper
            def __init__(self, x): self.x = x
            def __len__(self): return len(self.x)
            def __getitem__(self, i): return self.x[i]
        build_teacher_cache(_L(samples), out_path=TEACHER_CACHE, teacher=teacher, subsets_per_sample=6,
                            rel_lookup=rel_lookup, ent_lookup=ent_lookup, question_lookup=q_lookup)
        del teacher; import gc; gc.collect(); torch.cuda.empty_cache()
    print('cache entries:', len(torch.load(TEACHER_CACHE, map_location='cpu', weights_only=False)))

    B05 = f'{RESULTS}/b05_prism_llm'
    R05 = run_bench(B05, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['prism'],
                    extra=f'--dev-split 0.1 --prism-distill-epochs 10 --prism-rl-steps 500 --prism-teacher-cache {TEACHER_CACHE} --compare-ref prism --quiet')
    show(R05)
    f_llm = R05['summaries']['webqsp:test']['prism']['f1']; f_orc = R04['summaries']['webqsp:test']['prism']['f1']
    print(f'ΔF1 (LLM − oracle) = {(f_llm - f_orc)*100:+.2f} pp   | teacher = {R05["train_logs"]["prism@seed42"]["teacher"]}')
    gate(f_llm >= f_orc - 0.01, 'PRISM(LLM) ≥ PRISM(oracle) − 1 pp  → được viết "distilled from LLM"')
    if f_llm < f_orc - 0.03: print('WARN: KILL CRITERION: PRISM → future work, CGAR làm chính')
    PRISM_CKPT = f'{B05}/ckpt/prism_seed42.pt' if f_llm >= f_orc - 0.03 else PRISM_ORACLE_CKPT
    print('PRISM_CKPT =', PRISM_CKPT)

## 06 · Bảng chính với 3 seed và sweep Pareto
**Mục tiêu:** bảng duy nhất paper trích: mọi method cùng candidate/path/metric, CI, std seed, kiểm định cặp.
**Output kỳ vọng:** CGAR/PRISM dạng mean ± std (lần 11/2026: std 0.05–0.11 pp, rất ổn định); `comparisons` có Δ, CI, p bootstrap, p McNemar; `cgar_cost_sweep` (7 điểm max_paths × rel_threshold, cùng checkpoint, subset 600 câu), `prism_eps_sweep` (eps=0 × max_select 3/5/10 + eps>0); `pareto.png` có đường sweep.
Mốc Seminar 1 (GNN features thật, WebQSP): GNN-RAG-all 40.0/6.5, top-10 39.6/5.1, CGAR 40.7/5.1.
**Gate:** std theo seed < 1 pp F1. Thắng = Δ>0 với p<0.05 ở cùng cost; hoà = CI chứa 0 nhưng cost thấp hơn ≥20% (vẫn hợp lệ).

In [ ]:
B06 = f'{RESULTS}/b06_main' + ('_v3' if CGAR_FLAGS else '')   # thư mục theo biến thể được chọn ở 03 (b06_main cũ = gốc, dùng lại được)
R06 = run_bench(B06, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE],
                methods=['gnnrag_all','gnnrag_top10','gnnrag_top5','sbert_top10','sbert_top5','graphrag_fi','cgar'] + (['prism'] if RUN_PRISM else []),
                extra=f'--seeds 42 1337 2024 --sweep --dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s2-steps {CGAR_S2_STEPS} {CGAR_FLAGS} ' + (f'--prism-teacher-cache {TEACHER_CACHE} ' if RUN_PRISM else '')
                      + f'--compare-ref gnnrag_all --n-bootstrap 10000 --quiet')
show(R06); show_cmp(R06)
for ds, by_m in R06['seeds'].items():
    for m in (('cgar','prism') if RUN_PRISM else ('cgar',)):
        sd = by_m.get(m, {}).get('f1', {}); print(ds, m, 'F1 = %.2f ± %.2f' % (sd.get('mean',0)*100, sd.get('std',0)*100))
        gate(sd.get('std', 1) < 0.01, f'{ds} {m}: std F1 theo seed < 1 pp')
from kgqa_bench.metrics import matched_cost_table
for ds in R06['summaries']:
    print(ds, 'matched-cost vs cgar:', matched_cost_table(R06['summaries'][ds], 'cgar', tol=0.75))
for name in ('cgar_cost_sweep', 'cgar_beta_sweep', 'prism_eps_sweep'):
    sw = pd.DataFrame(R06['train_logs'].get(name, []))
    if len(sw): print('\n' + name); display(sw.round(4))
_c = R06['train_logs'].get('cgar_cost_sweep', []); _p = R06['train_logs'].get('prism_eps_sweep', [])
gate(len({round(p['cost'], 1) for p in _c if p['dataset'] == 'webqsp:test'}) >= 3, 'cgar_cost_sweep tạo ≥ 3 mức cost khác nhau (đường Pareto thật; β-sweep lần 11/2026 phẳng 9.96–9.99)')
if RUN_PRISM: gate(len({round(p['cost'], 1) for p in _p if p['dataset'] == 'webqsp:test' and p['eps'] > 0}) >= 2, 'eps PRISM có làm cost thay đổi (lần 11/2026: phẳng = 3 ở mọi eps)')
rep06 = open(f'{B06}/report.md').read(); print(rep06.split('## Adaptivity')[-1].split('## Sweeps')[0][:800] if '## Adaptivity' in rep06 else '')
from IPython.display import Image; display(Image(f'{B06}/pareto.png'))

## 06b · Confidence-gated escalation (offline, không inference thêm)
**Mục tiêu:** GNN-RAG+RA trong paper tăng Hit bằng cách *bổ sung* evidence; CGAR hiện chỉ *bớt* evidence và mất ~2 pp F1 so với all. Vì AURC của CGAR tốt nhất bảng, thử chính sách 2 tầng: trả lời bằng CGAR nếu `confidence ≥ τ`, ngược lại dùng toàn bộ path của gnnrag_all. Mọi điểm trên đường τ tính từ hai file records đã có (cùng qid), cộng điểm `oracle` (escalate đúng câu mà all tốt hơn) làm cận trên.
**Đọc:** đường chỉ có giá trị ở đoạn nằm **trên** các điểm top-K cùng cost (tham chiếu cùng qid). Khoảng cách tới `oracle` cho biết confidence head bắt được bao nhiêu phần gain khả dĩ.
**Gate:** tồn tại τ với cost ≤ 60% cost của all mà F1 ≥ F1(all) − 0.5 pp; và điểm đó cao hơn top-K gần nhất cùng cost.


In [ ]:
from kgqa_bench import escalation as E
from IPython.display import Image, display as _d
RE = E.run(B06, 'webqsp:test', primary='cgar', fallback='gnnrag_all', quality='f1', do_plot=True, verbose=False)
rows = pd.DataFrame(RE['rows']); display(rows[['tau','escalated_frac','cost','f1','coverage','hit_ctx']].round(4))
print('operating points:'); [print(f"  {k}: τ={v['tau']} escalate {v['escalated_frac']*100:.0f}% cost {v['cost']:.2f} F1 {v['f1']*100:.2f}") for k, v in RE['operating_points'].items()]
refs = pd.DataFrame(RE['references']).sort_values('cost'); display(refs[['method','cost','f1','hit_ctx']].round(4))
_d(Image(f"{B06}/escalation/escalation_webqsp_test_cgar_to_gnnrag_all.png"))
allrow = [r for r in RE['rows'] if r['tau'] == '+inf'][0]; orc = [r for r in RE['rows'] if r['tau'] == 'oracle'][0]
cheap = [r for r in RE['rows'] if r['tau'] not in ('oracle',) and r['cost'] <= 0.6 * allrow['cost']]
best = max(cheap, key=lambda r: r['f1']) if cheap else None
gate(best is not None and best['f1'] >= allrow['f1'] - 0.005, f"có τ với cost ≤ 60% all ({0.6*allrow['cost']:.2f}) đạt F1 ≥ all − 0.5 pp (best {best['f1']*100:.2f}@{best['cost']:.2f} vs all {allrow['f1']*100:.2f}@{allrow['cost']:.2f})" if best else 'không có điểm nào ≤ 60% cost')
near = min(RE['references'], key=lambda d: abs(d['cost'] - (best['cost'] if best else 0))) if (best and RE['references']) else None
gate(bool(near) and best['f1'] > near['f1'] + 0.005, f"điểm đó hơn top-K gần nhất cùng cost ({near['method'] if near else '-'}: {near['f1']*100:.2f}@{near['cost']:.2f})" if near else 'không có tham chiếu')
print(f"oracle gate: escalate {orc['escalated_frac']*100:.0f}% → F1 {orc['f1']*100:.2f} @ cost {orc['cost']:.2f}  (cận trên cho mọi gate trên 2 method này)")


## 06c · Phân tích offline A2–A4 (adaptivity thật, CI từng cặp, selective curve)
**A2** tách cost thấp do controller dừng sớm khỏi cost thấp do pool cạn: với mỗi câu tính số path có sẵn (cùng extraction) → `controller_stopped_early` = cost < min(n_paths, budget). Kèm F1 theo số ứng viên trong pool.
**A3** CI paired cho các cặp chưa có: CGAR vs sbert_top5, vs gnnrag_top5 (hiện chỉ có CI so với all).
**A4** đường selective answering (Hit-ctx theo coverage xếp theo confidence) cho mọi method có confidence — hình trung tâm của claim abstain.
**Gate:** tỉ lệ `controller_stopped_early` được báo cáo đúng nghĩa (nếu ≈0 thì không gọi CGAR là adaptive, trừ khi v3 learned-stop được chọn); CI CGAR − sbert_top5 không chứa 0 thì mới claim 'hơn SBERT'.


In [ ]:
from kgqa_bench import analysis as AN
RA = AN.run(B06, 'webqsp:test', eval_spec=WEBQSP_TE, budget=CGAR_BUDGET, primary='cgar',
            pairs=['cgar:sbert_top5','cgar:gnnrag_top5','cgar:gnnrag_top10','cgar:gnnrag_all'], n_bootstrap=10000, do_plot=True, verbose=False)
print(open(f"{B06}/analysis/analysis_webqsp_test.md").read())
from IPython.display import Image, display as _d; _d(Image(f"{B06}/analysis/selective_webqsp_test.png"))
cats = RA['adaptivity'].get('categories', {}); early = cats.get('controller_stopped_early', {}).get('frac', 0.0)
print(f"controller_stopped_early = {early*100:.1f}% | pool_limited = {cats.get('pool_limited',{}).get('frac',0)*100:.1f}% | full_budget = {cats.get('full_budget',{}).get('frac',0)*100:.1f}%")
gate(early >= 0.10 or not CGAR_FLAGS, 'controller tự dừng sớm ở ≥10% câu (adaptivity thật, không phải pool cạn)' if CGAR_FLAGS else 'bản gốc: adaptivity = pool cạn → ghi caveat, không claim adaptive')
cs = RA['pairwise'].get('cgar:sbert_top5', {}).get('f1', {}); b = cs.get('bootstrap', {})
if b: print(f"CGAR − sbert_top5 F1: Δ={b['delta']*100:+.2f} CI=[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}] p={b['p_value']:.4f}")
gate(bool(b) and b['ci_low'] > 0, 'CGAR > sbert_top5 với CI không chứa 0 (gain đến từ controller, không chỉ embedding)')
sel = RA['selective']; print({m: round(r['aurc'],3) for m, r in sel.items()})
gate('cgar' in sel and all(sel['cgar']['aurc'] <= r['aurc'] for r in sel.values()), 'AURC CGAR thấp nhất trong mọi method có confidence')


## 07 · Calibration và abstention
**Mục tiêu:** confidence của controller có dùng để từ chối trả lời không, và có hơn temperature scaling trên điểm GNN không.
**Output kỳ vọng:** max-GNN ECE 0.15–0.25; temp-scaled 0.05–0.10; CGAR/PRISM mục tiêu <0.08, AURC thấp nhất, sel_acc@80 > hit_ctx + 5 pp. Số ECE trong `PRISM_experiments.md` là placeholder, không dùng.
**Gate:** chỉ đánh giá ECE khi las_hit1 ≥ 0.2 (lần 11/2026: las_hit1 0.03 → ECE 0.06–0.08 của mọi phương pháp là tầm thường, isotonic GNN 0.024 thắng). AURC CGAR < top-10 là gate luôn hợp lệ.

In [ ]:
from kgqa_bench.data import load_rog_hf
from kgqa_bench.adapters import PrismVocab
from kgqa_bench.methods import PRISMMethod
from grl_rag import Embedder
from prism.config import CFG
from prism.experiments.exp4_calibration import run_exp4a_in_dist, run_exp4b_shift
B07 = f'{RESULTS}/b07_calibration'; os.makedirs(B07, exist_ok=True)

# ECE/AURC/sel_acc của mọi method đã có trong results của bước 06 (vs hit_ctx):
show(R06, cols=['hit_ctx','ece','ece_adaptive','mce','brier','aurc','sel_acc@80','sel_acc@50','overconf_rate'])

# Baseline công bằng (temp-scaled / isotonic trên điểm GNN), fit trên dev tách từ train:
emb = Embedder(model_name=EMB_MODEL)
from kgqa_bench.data import load_spec
tr = load_spec(WEBQSP_TR, cache_dir=DATA_CACHE); n_dev = int(len(tr)*0.1)
dev = type(tr)(tr.name, 'dev', tr.questions[-n_dev:], tr.source); tr.questions = tr.questions[:-n_dev]
te = load_spec(WEBQSP_TE, cache_dir=DATA_CACHE); cwq = load_spec(CWQ_TE, cache_dir=DATA_CACHE)
if RUN_PRISM:
    pm = PRISMMethod(emb, cfg=CFG, device=DEVICE).load(PRISM_CKPT, vocab_path=PRISM_CKPT + '.vocab.json')
    rep_a = run_exp4a_in_dist(pm.las, pm.samples_for(te), pm.samples_for(dev), CFG, out_dir=B07)
    rep_b = run_exp4b_shift(pm.las, pm.samples_for(dev), pm.samples_for(cwq), CFG, out_dir=B07)
base = R06['summaries']['webqsp:test'].get('prism', {}).get('las_hit1', 0.0)
if not RUN_PRISM:
    print('[skip] exp4 (PRISM) — nhánh CGAR-only; calibration CGAR đọc từ bảng trên (ECE/AURC/sel_acc vs hit_ctx)')
elif base < 0.2:
    print(f'WARN: las_hit1 = {base:.3f} < 0.2: ECE/Brier ở tỉ lệ dương thấp là tầm thường (dự đoán ~0 cho mọi câu cũng được ECE nhỏ). KHÔNG kết luận calibration trên pool này — chờ GNN features.')
else:
    gate(rep_a['PRISM (ours)']['ece'] < rep_a['Temp-scaled GNN']['ece'], 'PRISM ECE < temp-scaled GNN (in-dist)')
gate(R06['summaries']['webqsp:test']['cgar']['aurc'] < R06['summaries']['webqsp:test']['gnnrag_top10']['aurc'], 'AURC CGAR < top-10 (selective answering; không phụ thuộc tỉ lệ dương)')
from IPython.display import Image
for ds in ('webqsp_test','cwq_test'): display(Image(f'{B06}/reliability_{ds}.png'))

## 08 · LLM reader end-to-end và hallucination
**Mục tiêu:** retrieval tốt hơn có thành câu trả lời tốt hơn không; đo hallucination = `llm_unsupported_rate` (entity trả lời không có trong context) so với reader **không có context**.
**Output kỳ vọng:** no_retrieval: llm_hit thấp, unsupported cao; gnnrag_all cao nhất (Seminar 1: 83.7 Llama-2-7B); cgar/prism trong 1–2 pp của all với `n_tokens_prompt` thấp hơn ≥20%.
**Kết quả 11/2026 (Qwen2.5-7B, pool heuristic, 1000 câu đầu):** no_retrieval 0.476 > gnnrag_all 0.410 ≈ cgar 0.402 (p=0.48) > top5 0.203: context heuristic làm reader TỆ hơn không RAG, đúng pattern của paper tham chiếu. Đây là finding cho mục "RAG mitigates hallucination?": chất lượng retrieval quyết định. Kỳ vọng với GNN features: all ≈ 0.80+ > no_retrieval.
**Gate:** RAG(all) > no_retrieval; CGAR ≈ all (McNemar p > 0.05); unsupported_rate(cgar) < no_retrieval. Subset 1000 câu lấy ngẫu nhiên (`--subset-seed 0`), không lấy 1000 câu đầu.

In [ ]:
READERS = ['Qwen/Qwen2.5-7B-Instruct', 'NousResearch/Meta-Llama-3.1-8B-Instruct']
# 'mistralai/Mistral-7B-Instruct-v0.3' bỏ 11/2026 (~3h GPU, Llama đã cho cùng kết luận); thêm lại khi cần reader thứ 3
# 'openai/gpt-oss-20b' cần ~40GB VRAM bf16 — thêm vào khi có A100-80GB
MAX_ITEMS = 1000
R08 = {}
def reader_dir(rd, prompt):   # chat = thư mục cũ (dùng lại); grounded = b08g_*; v3 → hậu tố _v3 vì CGAR khác checkpoint
    return f"{RESULTS}/{'b08' if prompt == 'chat' else 'b08g'}_reader_{rd.split('/')[-1]}" + ('_v3' if CGAR_FLAGS else '')
for rd in READERS:
  for prompt in READER_PROMPTS:
    out = reader_dir(rd, prompt)
    R08[(rd, prompt)] = run_bench(out, evals=[WEBQSP_TE], methods=['gnnrag_all','gnnrag_top5','sbert_top5','cgar'] + (['prism'] if RUN_PRISM else []),
                        extra=f'--max-items {MAX_ITEMS} --subset-seed 0 --cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} ' + (f'--prism-ckpt {PRISM_CKPT} ' if RUN_PRISM else '')
                              + f'--reader hf:{rd}:{prompt} --no-retrieval-baseline --compare-ref gnnrag_all --quiet')
    print('\n#####', rd, '| prompt =', prompt); show(R08[(rd, prompt)], cols=['cost','n_tokens_prompt','hit_ctx','llm_hit','llm_hit1','llm_f1','llm_unsupported_rate','ece_llm','aurc_llm']); show_cmp(R08[(rd, prompt)])
    s = R08[(rd, prompt)]['summaries']['webqsp:test']
    d_rag = (s['gnnrag_all']['llm_hit'] - s['no_retrieval']['llm_hit']) * 100
    print(f'RAG(all) − no_retrieval = {d_rag:+.1f} pp llm_hit  (Qwen2.5-7B, pool heuristic 11/2026: −6.6 pp → RAG làm HẠI, đúng pattern paper tham chiếu PyPI/RAG)')
    gate(d_rag > 0, 'retrieval giúp reader hơn không-RAG (kỳ vọng PASS với GNN features, FAIL với pool heuristic → ghi thành finding)')
    gate(s['no_retrieval']['llm_hit'] < s['gnnrag_all']['llm_hit'] or s['gnnrag_top5']['llm_hit'] <= s['gnnrag_all']['llm_hit'] + 0.02, 'top5 ≤ all (nhiều context hơn không làm tệ hơn)')
    gate(s['cgar']['llm_unsupported_rate'] <= s['gnnrag_all']['llm_unsupported_rate'] + 0.01, f"CGAR không tăng unsupported_rate so với GNN-RAG-all ({s['cgar']['llm_unsupported_rate']:.3f} vs {s['gnnrag_all']['llm_unsupported_rate']:.3f}; so với no-RAG=1.0 là tầm thường)")
    gate(s['cgar']['llm_hit'] >= s['sbert_top5']['llm_hit'], 'CGAR ≥ sbert_top5 trên llm_hit')
    c = R08[(rd, prompt)]['comparisons']['webqsp:test'].get('cgar_vs_gnnrag_all:llm_hit', {})
    gate(bool(c) and (c['bootstrap']['delta'] > 0 or not c['significant_0.05']), 'CGAR ≥ hoặc tương đương GNN-RAG-all trên llm_hit')
# D1: prompt grounded có giảm unsupported mà không mất quá nhiều hit? (so cùng reader, cùng qid)
for rd in READERS:
    if (rd, 'chat') in R08 and (rd, 'grounded') in R08:
        a, g = R08[(rd, 'chat')]['summaries']['webqsp:test'], R08[(rd, 'grounded')]['summaries']['webqsp:test']
        for m in ('gnnrag_all', 'cgar'):
            print(f"{rd.split('/')[-1]:>28s} {m:>10s}: unsupported chat {a[m]['llm_unsupported_rate']:.3f} → grounded {g[m]['llm_unsupported_rate']:.3f} | llm_hit {a[m]['llm_hit']:.3f} → {g[m]['llm_hit']:.3f}")
        gate(g['cgar']['llm_unsupported_rate'] < a['cgar']['llm_unsupported_rate'] - 0.02, f"{rd.split('/')[-1]}: prompt grounded giảm unsupported của CGAR ≥ 2 pp")
        gate(g['cgar']['llm_hit'] >= a['cgar']['llm_hit'] - 0.03, f"{rd.split('/')[-1]}: prompt grounded không làm CGAR mất quá 3 pp llm_hit")

## 08b · Hallucination / groundedness (offline trên records của 08)
**Mục tiêu:** tách *faithfulness* (câu trả lời có được graph evidence hỗ trợ) khỏi *correctness*. `llm_unsupported_rate` cũ chỉ kiểm tra entity có xuất hiện trong context (false negative kiểu "Alice works_at X, X located_in London → born in London"; alias bị tính là bịa).
**Cách đo:** mỗi entity trả lời = 1 claim → tầng 1 kiểm tra graph (endpoint của path topic→entity / chỉ là node trung gian / có trong subgraph nhưng không được retrieve / không có); tầng 2 so chuỗi quan hệ có hướng của path với reference proofs (topic→gold trong full subgraph, chỉ dùng lúc đánh giá) → SUPPORTED / UNSUPPORTED / UNCERTAIN (+ CONTRADICTED nếu bật judge).
**Output:** `results_gnn/b08_reader_*/hallucination/`: `report_*.md`, `hallucination.json`, `claims_*.jsonl`, `risk_coverage_*.png`, `review_sample_*.csv` (200 dòng cho human review, ưu tiên câu hai method bất đồng).
**Gate:** CGAR không tăng question-HR so với gnnrag_all (paired, p ≥ 0.05); AURC-hallucination của CGAR < gnnrag_all (confidence xếp hạng được hallucination). Grounded-correct rate báo cáo kèm llm_hit để tránh thắng nhờ abstain.


In [ ]:
from kgqa_bench import hallucination as H
R08H = {}
for rd in READERS:
  for prompt in READER_PROMPTS:
    out = reader_dir(rd, prompt)
    if not os.path.exists(f'{out}/results.json'): print('[skip]', out); continue
    res = H.run(out, [WEBQSP_TE], methods=['gnnrag_all','gnnrag_top5','sbert_top5','cgar'] + (['prism'] if RUN_PRISM else []),
                ref='gnnrag_all', review_n=200, n_bootstrap=10000, verbose=False)
    dsk = list(res['datasets'])[0]; S = res['datasets'][dsk]['summaries']; C = res['datasets'][dsk]['comparisons']
    R08H[(rd, prompt)] = res
    print('\n#####', rd, '| prompt =', prompt)
    display(pd.DataFrame(S).T[['n_answered','abstention_rate','entity_unsupported_rate','claim_hallucination_rate','question_hallucination_rate',
                               'question_hallucination_rate_strict','graph_proof_support_rate','uncertain_rate','llm_hit','grounded_correct_rate',
                               'grounded_answer_f1','correct_ungrounded_rate','aurc_hallucination','aurc_incorrect']].astype(float).round(3))
    print('error taxonomy:'); display(pd.DataFrame({m: s['error_taxonomy'] for m, s in S.items()}).fillna(0).astype(int))
    c = C.get('cgar', {}).get('hallucinated', {}); b = c.get('bootstrap', {})
    if b: print(f"CGAR − all question-HR: Δ={b['delta']*100:+.2f} pp CI=[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}] p={b['p_value']:.4f} McNemar p={c['mcnemar']['p_value']:.4f}")
    gate(bool(b) and (b['delta'] <= 0 or b['p_value'] >= 0.05), 'CGAR không tăng question-hallucination-rate so với GNN-RAG-all (paired)')
    gate('aurc_hallucination' in S.get('cgar', {}) and S['cgar']['aurc_hallucination'] < S['gnnrag_all']['aurc_hallucination'],
         f"AURC-hallucination CGAR {S.get('cgar',{}).get('aurc_hallucination')} < GNN-RAG-all {S['gnnrag_all'].get('aurc_hallucination')} (confidence xếp hạng được hallucination)")
    gate(S['cgar']['grounded_correct_rate'] >= S['sbert_top5']['grounded_correct_rate'], 'grounded-correct CGAR ≥ sbert_top5')
    from IPython.display import Image, display as _d; _d(Image(f"{out}/hallucination/risk_coverage_{dsk.replace(':','_')}.png"))
print('review_sample_*.csv → chia cho 2 người gán nhãn, đo precision/recall của detector trước khi viết claim')
# escalation ở mức reader: cùng 1000 qid, chuyển sang gnnrag_all khi CGAR conf < τ → llm_hit / unsupported / token thật
from kgqa_bench import escalation as E
for rd in READERS:
    out = reader_dir(rd, 'chat')
    if not os.path.exists(f'{out}/results.json'): continue
    RE8 = E.run(out, 'webqsp:test', primary='cgar', fallback='gnnrag_all', quality='llm_hit', refs=['gnnrag_top5','sbert_top5'], do_plot=False, verbose=False)
    print('\n##### escalation @ reader', rd); display(pd.DataFrame(RE8['rows'])[['tau','escalated_frac','cost','n_tokens_prompt','f1','llm_hit','llm_f1','llm_unsupported_rate']].round(4))


## 09 · Ablation
**Mục tiêu:** mỗi thành phần giữ lại phải có bằng chứng; thành phần không đóng góp → bỏ hoặc báo cáo negative finding.
**Output kỳ vọng:** CGAR S1 vs S1+S2: ΔF1 trong ±0.5, không ý nghĩa; `rel_threshold 0.0`: cost −20…−40%, F1 −<1 pp; PRISM no-RL vs full: Δ nhỏ; E (no info-stop): cost tăng, F1 không tăng; D (no counterfactual): F1 giảm rõ nếu claim đúng.
**Gate:** mỗi hàng có Δ kèm CI. RL không giúp ở cả hai → viết thành finding độc lập.

In [ ]:
ABL = {}
base = dict(train=WEBQSP_TR, evals=[WEBQSP_TE])
V = '_v3' if CGAR_FLAGS else ''
# (a) RL: bản chính là S1; S2 là ablation (b03_cgar cũ có sẵn ckpt S2 → chỉ eval, không train lại)
_s2 = f'{B03}/ckpt/cgar_s2_seed42.pt'
ABL['cgar_base_s1']  = run_bench(f'{RESULTS}/b09_cgar_noRL',  methods=['cgar'],  extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s2-steps 0 --compare-ref cgar --quiet', **base)
ABL['cgar_base_rl']  = (run_bench(f'{RESULTS}/b09_cgar_withRL', methods=['cgar'], extra=f'--cgar-ckpt {_s2} --cgar-max-paths {CGAR_BUDGET} --compare-ref cgar --quiet', evals=[WEBQSP_TE]) if os.path.exists(_s2)
                        else run_bench(f'{RESULTS}/b09_cgar_withRL', methods=['cgar'], extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s2-steps 300 --compare-ref cgar --quiet', **base))
# (b) nút cost cố định trên checkpoint chính
ABL['cgar_thr0']     = run_bench(f'{RESULTS}/b09_cgar_thr0{V}',   methods=['cgar'],  extra=f'--cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} --cgar-rel-threshold 0.0 --compare-ref cgar --quiet', evals=[WEBQSP_TE])
ABL['cgar_thr-1']    = run_bench(f'{RESULTS}/b09_cgar_thr-1{V}',  methods=['cgar'],  extra=f'--cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} --cgar-rel-threshold -1.0 --compare-ref cgar --quiet', evals=[WEBQSP_TE])
# (c) từng thành phần v3 riêng lẻ (B1 learned stop | B2 nhãn proof | B3 conf=hit) và cả ba
for tag, fl in [('lstop', '--cgar-learned-stop'), ('proof', '--cgar-rel-label proof'), ('hitconf', '--cgar-conf-target hit'),
                ('v3_all', '--cgar-learned-stop --cgar-rel-label proof --cgar-conf-target hit')]:
    ABL[f'cgar_{tag}'] = run_bench(f'{RESULTS}/b09_cgar_{tag}', methods=['cgar'], extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s2-steps 0 {fl} --compare-ref cgar --quiet', **base)
# (d) learned-stop threshold sweep trên checkpoint v3 (không train lại) → đường cost/F1 do controller tự quyết
if os.path.exists(f'{RESULTS}/b09_cgar_lstop/ckpt/cgar_s1_seed42.pt'):
    for th in (0.3, 0.7):
        ABL[f'cgar_lstop_th{th}'] = run_bench(f'{RESULTS}/b09_cgar_lstop_th{th}', methods=['cgar'], evals=[WEBQSP_TE],
                                              extra=f"--cgar-ckpt {RESULTS}/b09_cgar_lstop/ckpt/cgar_s1_seed42.pt --cgar-max-paths {CGAR_BUDGET} --cgar-learned-stop --cgar-stop-threshold {th} --compare-ref cgar --quiet")
if RUN_PRISM: ABL['prism_noRL'] = run_bench(f'{RESULTS}/b09_prism_noRL',  methods=['prism'], extra=f'--dev-split 0.1 --prism-rl-steps 0 --prism-teacher-cache {TEACHER_CACHE} --compare-ref prism --quiet', **base)
rows = {'cgar_main(06)': R06['summaries']['webqsp:test']['cgar']}
if RUN_PRISM: rows['prism_full'] = R06['summaries']['webqsp:test']['prism']
for k, R in ABL.items(): rows[k] = list(R['summaries']['webqsp:test'].values())[0]
display(pd.DataFrame(rows).T[['f1','cov','prec','hit_ctx','cost','ece','aurc','sel_acc@80']].round(3))

# Δ với CI (paired) giữa biến thể và bản đầy đủ, trên cùng câu hỏi
from kgqa_bench.metrics import compare_records
def recs(out, method): return [json.loads(l) for l in open(f'{out}/records_webqsp_test_{method}.jsonl')]
base_s1 = recs(f'{RESULTS}/b09_cgar_noRL', 'cgar')
for name in ['cgar_base_rl', 'cgar_lstop', 'cgar_proof', 'cgar_hitconf', 'cgar_v3_all']:
    out = {'cgar_base_rl': f'{RESULTS}/b09_cgar_withRL'}.get(name, f'{RESULTS}/b09_{name}')
    if not os.path.exists(f'{out}/records_webqsp_test_cgar.jsonl'): continue
    for met in ('f1', 'hit_ctx'):
        c = compare_records(recs(out, 'cgar'), base_s1, met); b = c['bootstrap']
        print(f"{name} − base_s1 [{met}]: Δ={b['delta']*100:+.2f} CI=[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}] p={b['p_value']:.3f}")
gate(rows['cgar_base_rl']['f1'] <= rows['cgar_base_s1']['f1'] + 0.002, 'RL (S2) không cải thiện F1 so với S1 → S1 là phương pháp chính (B4)')
if 'cgar_proof' in rows: gate(rows['cgar_proof']['prec'] >= rows['cgar_base_s1']['prec'], 'nhãn proof (B2) không giảm Prec so với nhãn entity')
if 'cgar_hitconf' in rows: gate(rows['cgar_hitconf']['ece'] < rows['cgar_base_s1']['ece'], 'conf-target=hit (B3) giảm ECE')
if 'cgar_lstop' in rows: gate(rows['cgar_lstop']['cost'] < rows['cgar_base_s1']['cost'] - 0.1 and rows['cgar_lstop']['hit_ctx'] >= rows['cgar_thr0']['hit_ctx'], 'learned stop (B1) giảm cost và giữ hit_ctx tốt hơn rel_threshold cố định')

# PRISM controller ablations D/E/F/H (exp3) trên checkpoint đã train
if RUN_PRISM:
    from prism.experiments.exp3_ablation import run_exp3
    from prism.las import LAS
    pm = PRISMMethod(emb, cfg=CFG, device=DEVICE).load(PRISM_CKPT, vocab_path=PRISM_CKPT + '.vocab.json')
    las_distill = LAS(CFG).to(DEVICE); las_distill.load_state_dict(torch.load(f'{B05}/ckpt/prism_distill_best.pt', map_location=DEVICE))
    exp3 = run_exp3(pm.las, las_distill, pm.samples_for(te), CFG, out_dir=f'{RESULTS}/b09_prism_exp3')
else:
    print('[skip] exp3 (PRISM) — nhánh CGAR-only')

## 10 · Transfer giữa dataset và drift
**Mục tiêu:** kiểm tra claim inductive của PRISM bằng drift có ý nghĩa (quan hệ mới, điểm retriever nhiễu) áp dụng **giống nhau cho cả CGAR và PRISM** ở cấp dataset.
**Output kỳ vọng:** mode `relation_oov` 50% → giảm <10 pp là tốt; `score_noise` làm cả hai giảm, chênh độ dốc là kết quả; transfer CWQ (bước 06) giữ thứ hạng so với top-K.
**Gate:** có đường cong relation_oov cho cả hai method trên cùng dataset. Không dùng mode `id` (phẳng theo xây dựng) làm hình chính.

In [ ]:
from kgqa_bench.data import drift_dataset
from kgqa_bench.methods import CGARMethod
from kgqa_bench.metrics import summarize_records
B10 = f'{RESULTS}/b10_drift'; os.makedirs(B10, exist_ok=True)
cm = CGARMethod(emb, device=DEVICE, max_paths=CGAR_BUDGET, learned_stop=('--cgar-learned-stop' in CGAR_FLAGS)).load(CGAR_CKPT)
methods_drift = [cm] + ([PRISMMethod(emb, cfg=CFG, device=DEVICE).load(PRISM_CKPT, vocab_path=PRISM_CKPT + '.vocab.json')] if RUN_PRISM else [])
sub = te.subset(600, seed=0)   # 600 câu đủ cho đường cong; bỏ .subset để chạy full
rows = []
for mode in ['relation_oov', 'score_noise', 'mixed']:
    for pct in [0.0, 0.1, 0.3, 0.5]:
        d = drift_dataset(sub, pct, mode, seed=0).annotate_hops()
        for m in methods_drift:
            s = summarize_records(m.run(d, verbose=False), n_bootstrap=200)
            rows.append(dict(mode=mode, pct=pct, method=m.name, f1=s['f1'], cov=s['cov'], cost=s['cost'], ece=s.get('ece')))
df = pd.DataFrame(rows); df.to_csv(f'{B10}/drift_curves.csv', index=False)
display(df.pivot_table(index=['mode','pct'], columns='method', values='f1').round(3))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mode in zip(axes, ['relation_oov','score_noise','mixed']):
    for m in [x.name for x in methods_drift]:
        q = df[(df['mode']==mode)&(df['method']==m)]; ax.plot(q['pct']*100, q['f1']*100, 'o-', label=m)
    ax.set_title(mode); ax.set_xlabel('% drift'); ax.set_ylabel('F1'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f'{B10}/drift_curves.png', dpi=150); plt.show()
d0 = df[(df['mode']=='relation_oov')]
drop = {m: d0[(d0['method']==m)&(d0['pct']==0)]['f1'].item() - d0[(d0['method']==m)&(d0['pct']==0.5)]['f1'].item() for m in [x.name for x in methods_drift]}
print('F1 drop @50% relation_oov:', {k: round(v*100,2) for k,v in drop.items()})
if RUN_PRISM: gate(drop['prism'] <= drop['cgar'], 'PRISM giảm không nhiều hơn CGAR dưới relation_oov')
else: gate(drop['cgar'] < 0.10, f"CGAR giảm {drop['cgar']*100:.1f} pp @50% relation_oov (< 10 pp: SBERT vẫn hiểu quan hệ lạ)")

## 11 · Report và checklist claim
**Mục tiêu:** mọi con số trong paper truy ngược được về một `results.json` và một lệnh; mỗi claim chỉ được viết khi gate tương ứng đã qua.
**Tối thiểu cho Seminar 2:** bước 01, 02, 03, 06 (một seed), 08 (một reader). **EMNLP:** toàn bộ 00–11.

In [ ]:
!python -m kgqa_bench.cli report --results $B06 > /dev/null
print(open(f'{B06}/report.md').read()[:4000])

# Checklist claim ↔ gate (tự động từ kết quả đã có)
claims = []
def have(path): return os.path.exists(path)
s06 = R06['summaries']['webqsp:test']; c06 = R06['comparisons']['webqsp:test']
cg = c06.get('cgar_vs_gnnrag_all:f1', {});
claims.append(('Giảm cost ở cùng chất lượng (06)', bool(cg) and (cg['bootstrap']['delta'] >= 0 or not cg['significant_0.05']) and s06['cgar']['cost'] <= 0.8*s06['gnnrag_all']['cost']))
G = globals()
claims.append(('Confidence dùng được để abstain (07)', s06['cgar']['aurc'] < s06['gnnrag_top10']['aurc'] and ((not RUN_PRISM) or ('rep_a' in G and s06['prism'].get('las_hit1', 0) >= 0.2 and rep_a['PRISM (ours)']['ece'] < rep_a['Temp-scaled GNN']['ece']))))
claims.append(('Chọn lọc không tăng unsupported (proxy cũ) so với GNN-RAG-all (08)', all(R['summaries']['webqsp:test']['cgar']['llm_unsupported_rate'] <= R['summaries']['webqsp:test']['gnnrag_all']['llm_unsupported_rate'] + 0.01 for k, R in G.get('R08', {}).items() if k[1] == 'chat') if G.get('R08') else False))
claims.append(('RAG giúp reader hơn không-RAG (08)', any(R['summaries']['webqsp:test']['gnnrag_all']['llm_hit'] > R['summaries']['webqsp:test']['no_retrieval']['llm_hit'] for R in G.get('R08', {}).values()) if G.get('R08') else False))
claims.append(('Distilled from LLM (05)', 'R05' in G and R05['train_logs']['prism@seed42']['teacher'] != 'oracle_heuristic' and f_llm >= f_orc - 0.01))
claims.append(('PRISM LAS học được, không ở mức chance (04)', bool(RUN_PRISM) and s06.get('prism', {}).get('las_hit1', 0) >= G.get('PRISM_CHANCE', 1/32) + 0.10))
claims.append(('Controller adaptive thật (06: sweep cost không phẳng)', len({round(p['cost'],1) for p in R06['train_logs'].get('cgar_cost_sweep', []) if p['dataset']=='webqsp:test'}) >= 3))
claims.append(('Inductive / robust to drift (10)', 'drop' in G and have(f'{B10}/drift_curves.csv') and (drop['prism'] <= drop['cgar'] if RUN_PRISM else drop['cgar'] < 0.10)))
claims.append(('RL không cần thiết (09)', 'ABL' in G and 'cgar_base_rl' in ABL and ABL['cgar_base_rl']['summaries']['webqsp:test']['cgar']['f1'] <= ABL['cgar_base_s1']['summaries']['webqsp:test']['cgar']['f1'] + 0.002))
# --- cải tiến 11/2026 ---
_re = G.get('RE'); _allrow = [r for r in _re['rows'] if r['tau'] == '+inf'][0] if _re else None
claims.append(('Escalation theo confidence đạt F1 ≈ all ở ≤ 60% cost (06b)', bool(_re) and any(r['tau'] not in ('oracle',) and r['cost'] <= 0.6*_allrow['cost'] and r['f1'] >= _allrow['f1'] - 0.005 for r in _re['rows'])))
_ra = G.get('RA'); _cs = (_ra or {}).get('pairwise', {}).get('cgar:sbert_top5', {}).get('f1', {}).get('bootstrap', {})
claims.append(('CGAR > sbert_top5 với CI không chứa 0 (06c)', bool(_cs) and _cs.get('ci_low', -1) > 0))
claims.append(('Adaptivity do controller, không do pool (06c)', bool(_ra) and (_ra.get('adaptivity', {}).get('categories', {}).get('controller_stopped_early', {}).get('frac', 0) >= 0.10)))
_h = G.get('R08H', {})
def _hok(res):
    d = list(res['datasets'].values())[0]; c = d['comparisons'].get('cgar', {}).get('hallucinated', {}).get('bootstrap', {})
    return bool(c) and (c['delta'] <= 0 or c['p_value'] >= 0.05)
claims.append(('CGAR không tăng question-hallucination so với all (08b, graph-grounded)', bool(_h) and all(_hok(r) for r in _h.values())))
claims.append(('Prompt grounded giảm unsupported của CGAR ≥ 2 pp (08, D1)', any((rd, 'grounded') in R08 and (rd, 'chat') in R08 and R08[(rd,'grounded')]['summaries']['webqsp:test']['cgar']['llm_unsupported_rate'] < R08[(rd,'chat')]['summaries']['webqsp:test']['cgar']['llm_unsupported_rate'] - 0.02 for rd in READERS) if G.get('R08') else False))
claims.append(('CWQ chạy trên pool GNN (C2)', CWQ_TE.startswith('gnnrag:')))
for name, ok in claims: print(('[OK] ' if ok else '[NO] ') + name + ('' if ok else '  → hạ xuống Discussion/Limitations'))
print('\nMỗi claim [OK] viết kèm Δ, 95% CI, p lấy từ comparisons của bước tương ứng. KHÔNG dùng số placeholder trong PRISM_experiments.md / CGAR_abstract_v1.md.')